# Lab 5: Bayesian Classification
### COSC 426: Fall 2025, Colgate University

## Part 1: Build a unigram model

#### Part 1.1

Start by understanding what is happening when you initilize a UnigramModel object

In [5]:
from UnigramModel import UnigramModel
import util
import math
import pandas as pd
import numpy as np

sample_model = UnigramModel(tokenize = util.nltk_tokenize,
                            tokenizer_kwargs = {},
                            vocab = util.get_vocab('data/glove_vocab.txt'),
                            unk_token = '[UNK]',
                            train_paths = ['data/sample-alice.txt'],
                            smooth = 'add-0.1',
                            lower = True
                           )                    

**What are the parameters required to initialize a `UnigramModel` object and how are these parameters used in the `UnigramModel` class?**


- tokenize function & tokenizer_kwargs (args for tokenize function): preprocess to split text into tokens
- vocab: set of words we want the model to recognize
- unknown token: replace tokens not in vocab with unk token
- train file path: path to the training file
- smooth string: what smooth method we are using to get the smoothed probability
- lower boolean: if preprocessing only ouputs lowercase tokens

#### Part 1.2

Verify your implementation of `get_prob` and `evaluate` with the code below. 

*Hint: Running into errors `get_probs`? Think carefully about when are where preprocessing is being applied in the pipeline, and what the expected input for `get_probs` is. Also think about how you want to handle word to not exist in training data and/or not in vocab*

In [6]:
## print prob of words
expected_probs = {'rabbit': 0.00010176146615934851,
                  'it': 0.0002755005547240899,
                  '[UNK]': 0.00017622107554423767,
                  'Alice': 0.00017622107554423767,
                  'Rabbit': 0.00017622107554423767,
                  'maze': 2.4819869794963054e-06
                 }

for word in expected_probs:
    if sample_model.get_prob(word) != expected_probs[word]:
        print(word, '\t incorrect')
        print('expected', expected_probs[word])
        print('got', sample_model.get_prob(word))
        print()
        

## Create paths and then load it
sample_model.evaluate(datafpath= 'data/sample-alice.txt',
                      predfpath = 'predictions/my_sample_preds_alice.tsv')


correct_df = pd.read_csv('predictions/sample_preds_alice.tsv', sep='\t')
my_df = pd.read_csv('predictions/my_sample_preds_alice.tsv', sep='\t')

# Does element wise comparison
print('Are dfs same?', correct_df.equals(my_df))

Are dfs same? True


## Part 2: Implement building blocks of a Naive Bayesian Classifier

Implement the building blocks for a Bayesian classifier. Here is a function that might be useful. 

### Part 2.1: Describe your approach

- unigram language model
- method to compute likelihood: Together with the unigram model, which is trained with text from each specific class, the likelihood $P(text∣Class)$ is calculated.
- method to compute prior: With this, we get $P(Class)$.

The classifier will give the final conclusion based on $$P(Class∣text) \stackrel{\propto}{\sim} P(text∣Class)P(Class)$$

### Part 2.2 Implement functions

In [7]:
import pandas as pd
def summarize(fname:str, aggregrate_type:str, aggregrate_col:str, groupby_cols:list, delimiter='\t'):
    """
    Args:
        fname: fpath to tsv/ csv file
        aggregate_type: mean or sum

        aggrefate_col: the column with values you want to aggregate over

        groupby_cols: the columns with the groups. 

    Returns:
        Pandas Dataframe with as many rows as unique group combinations. The values of rows in each group is either summed together or averaged depending on the aggregate_type. 

    """
    dat = pd.read_csv(fname, sep=delimiter)

    summ = dat.groupby(groupby_cols).agg({aggregrate_col: aggregrate_type}).reset_index()

    return summ

#### **Calculating the likelihood**

Start by calculating the likelihood of some text given models trained on text from different classes --- i.e., $P(text \mid model=class1)$, $P(text \mid model=class2)$, etc

*Hint: Think about why `summarize` function provided is useful* 

In [8]:
def get_likelihood(models_dict, eval_fpath, class_label):
    """
    Params:
        models_dict: keys are classes and values are the models trained on the classes
        eval_fpath: the file models should be evaluated on
        class_label: the correct class label for sequences in the file 

    Returns:
        A Dataframe with the following columns: 
            sentid: id of the sentence
            model: the model being used to generate the likelihood
            likelihood: the sum of log probability across all the words in the sequence
            target_class: same as class_label
    """
    results = []

    for model_name, model in models_dict.items():
        test_data = model.preprocess([eval_fpath])

        for sentid, sent in enumerate(test_data):
            if sent == []:
                continue
            log_likelihood = 0.0
            for word in sent:
                prob = model.get_prob(word)
                if prob > 0:
                    log_likelihood += math.log2(prob) 
                else:
                    log_likelihood = float('-inf')
                    break

            results.append({
                "sentid": sentid,
                "model": model_name,
                "likelihood": log_likelihood,
                "target_class": class_label
            })

    return pd.DataFrame(results)
        

Once you've implemented this function, verify that your output matches the expected output below. 

In [9]:
sample_models = {
    'alice': UnigramModel(tokenize = util.nltk_tokenize,
                            tokenizer_kwargs = {},
                            vocab = util.get_vocab('data/glove_vocab.txt'),
                            unk_token = '[UNK]',
                            train_paths = ['data/sample-alice.txt'],
                            smooth = 'add-0.1',
                            lower = True
                           ),
    'sherlock': UnigramModel(tokenize = util.nltk_tokenize,
                            tokenizer_kwargs = {},
                            vocab = util.get_vocab('data/glove_vocab.txt'),
                            unk_token = '[UNK]',
                            train_paths = ['data/sample-sherlock.txt'],
                            smooth = 'add-0.1',
                            lower = True
                           )
}

my_df = get_likelihood(sample_models, 'data/sample-lookingglass.txt', 'alice').reset_index(drop=True)
correct_df = pd.read_csv('predictions/sample-likelihood.tsv', sep='\t').reset_index(drop=True)

#using this instead of equal because of floating point imprecision
print('Printing proportion of matched values across correct_df and my_df\n')
print('likelihood', np.isclose(my_df['likelihood'], correct_df['likelihood']).sum()/len(my_df)) 
for col in ['sentid', 'model', 'target_class']:
    print(col, (my_df[col] == correct_df[col]).sum()/len(my_df))

Printing proportion of matched values across correct_df and my_df

likelihood 1.0
sentid 1.0
model 1.0
target_class 1.0


#### Calculating the prior

In [10]:
def get_prior(data: dict, tokenizer) -> dict:
    """
    Args:
        data: dictionary where keys are the classes, and values are filepaths to the class specific data

    Returns:
        Dictionary with prior probability for each class, which is the number of words in the class divided by the total number of words across all classes. 
        What counts as a word is determined by the tokenizer

    """
    word_counts = {}
    total_words = 0

    for class_label, fpaths in data.items():

        class_word_count = 0
        for fpath in fpaths:
            with open(fpath, 'r') as f:
                for line in f:
                    tokens = tokenizer(line)
                    class_word_count += len(tokens)

        word_counts[class_label] = class_word_count
        total_words += class_word_count

    priors = {cls: count / total_words for cls, count in word_counts.items()}
    return priors


In [11]:
import nltk
dat_dict = {'sherlock': ['data/sample-sherlock.txt'],
            'alice': ['data/sample-alice.txt']}

correct_prior = {'sherlock': 0.4423076923076923, 'alice': 0.5576923076923077}
get_prior(dat_dict, nltk.word_tokenize) #pass in tokenizer

{'sherlock': 0.4423076923076923, 'alice': 0.5576923076923077}

#### Compute posterior

*Hint: Think about why `summarize` function provided is useful*

In [12]:
def get_posterior(models_dict, eval_fpath, class_label, prior_dict):
    """
    Args:
        Dictionary where keys are classes and values are the models trained on the classes

        eval_fpath: the file models should be evaluated on. 

        class_label: the label of the file that models are evaluated on

        prior_dict: prior probabilities of classes

    Returns:
        A Dataframe with the following columns: sentid, model, likelihood, class. 

    If you set eval_fpath to sample_reviews_test_positive.txt, you should get a dataframe that looks like this. (Its ok if you end up having additional columns)

   sentid model     likelihood   class        prior      posterior
       0  positive  -100.975898  positive    -0.736966   -101.712864
       1  positive  -100.941133  positive    -0.736966   -101.678099
       0  negative  -101.938780  positive    -1.321928   -103.260708
       1  negative  -101.938780  positive    -1.321928   -103.260708


    """
    results = []
    for model_name, model in models_dict.items():
        if isinstance(eval_fpath, list):
            test_data = model.preprocess(eval_fpath)
        else:
            test_data = model.preprocess([eval_fpath])

        for sentid, sent in enumerate(test_data):
            if not sent:
                continue 

            log_likelihood = 0.0
            for word in sent:
                prob = model.get_prob(word)
                if prob > 0:
                    log_likelihood += math.log2(prob)
                else:
                    log_likelihood = float('-inf')
                    break

            prior_prob = prior_dict.get(model_name, 1e-10)  # avoid log(0)
            log_prior = math.log2(prior_prob)

            log_posterior = log_likelihood + log_prior

            results.append({
                'sentid': sentid,
                'model': model_name,
                'likelihood': log_likelihood,
                'class': class_label,
                'prior': log_prior,
                'posterior': log_posterior
            })
    df = pd.DataFrame(results)

    return df

In [13]:
my_df = get_posterior(sample_models, 
                    'data/sample-lookingglass.txt',
                    'alice',
                     get_prior(dat_dict, nltk.word_tokenize)).reset_index(drop=True)

correct_df = pd.read_csv('predictions/sample-posterior.tsv', sep='\t').reset_index(drop=True)
print('posterior', np.isclose(my_df['posterior'], correct_df['posterior']).sum()/len(my_df)) 
          

posterior 1.0


#### Implement classify

In [14]:
def classify(posterior):
    """
    Args:
        Dataframe with posterior probabilities

    Returns: 
        Dataframe where each sentence id is associated with a prediction. 
    """

    # converts the data from long to wide
    classes = posterior['model'].unique()
    wide_df = posterior.pivot(index=['sentid', 'class'],
                              columns=['model'],
                              values='posterior').reset_index()    
    #Finish the rest of the function 

    posterior_cols = [col for col in wide_df.columns if col not in ['sentid', 'class']]
    wide_df['pred'] = wide_df[posterior_cols].idxmax(axis=1)
    
    return wide_df[['sentid', 'class', 'pred']]


In [15]:
posterior = get_posterior(sample_models, 
            'data/sample-lookingglass.txt',
            'alice',
             get_prior(dat_dict, nltk.word_tokenize)).reset_index(drop=True)
my_df = classify(posterior).reset_index(drop=True)

correct_df = pd.read_csv('predictions/sample-classify.tsv', sep='\t').reset_index(drop=True)
print('pred', (my_df['pred']==correct_df['pred']).sum()/len(my_df)) 

pred 1.0


#### Compute accuracy

In [16]:
def analyze(models_dict, eval_dict, prior_dict):
    """
    Args:
        models_dict: keys are classes, values are models trained on data from the class. 

        eval_dict: keys are classes, values are fpaths to evaluation data where the correct label is the class associated with the key

        prior_dict: keys are classes, values are prior probabilties of the classes. 

    Returns:
        Float which is the accuracy of the predictions across all classes

    """
    all_preds = []
    all_targets = []

    for class_label, fpath in eval_dict.items():
        posterior = get_posterior(models_dict, fpath, class_label, prior_dict).reset_index(drop=True)
        pred_df = classify(posterior).reset_index(drop=True)
        all_preds.extend(pred_df['pred'].tolist())
        all_targets.extend(pred_df['class'].tolist())

    correct = sum(p == t for p, t in zip(all_preds, all_targets))
    accuracy = correct / len(all_preds) if all_preds else 0.0

    return accuracy

In [17]:
sample_models = {
    'alice': UnigramModel(tokenize = util.nltk_tokenize,
                            tokenizer_kwargs = {},
                            vocab = util.get_vocab('data/glove_vocab.txt'),
                            unk_token = '[UNK]',
                            train_paths = ['data/sample-alice.txt'],
                            smooth = 'add-0.1',
                            lower = True
                           ),
    'sherlock': UnigramModel(tokenize = util.nltk_tokenize,
                            tokenizer_kwargs = {},
                            vocab = util.get_vocab('data/glove_vocab.txt'),
                            unk_token = '[UNK]',
                            train_paths = ['data/sample-sherlock.txt'],
                            smooth = 'add-0.1',
                            lower = True
                           )
}

## for simplicity making eval the same as train
sample_eval = {
    'alice': ['data/sample-lookingglass.txt'],
    'sherlock': ['data/sample-sherlock.txt']
}

sample_prior = get_prior({'sherlock': ['data/sample-sherlock.txt'],
                          'alice': ['data/sample-alice.txt']}, nltk.word_tokenize)


target_acc = 92.857

analyze(sample_models, sample_eval, sample_prior)


0.9285714285714286

## Part 3: Build Naive Bayesian Sentiment Classifier
Add as many code and markdown chunks as is helpful

In [18]:
import os

vocab = util.get_vocab("data/glove_vocab.txt")
unk_token = "[UNK]"

train_pos_dir = "aclImdb/train/pos"
train_neg_dir = "aclImdb/train/neg"
test_pos_dir  = "aclImdb/test/pos"
test_neg_dir  = "aclImdb/test/neg"

train_pos_files = [os.path.join(train_pos_dir, f) for f in os.listdir(train_pos_dir) if f.endswith(".txt")]
train_neg_files = [os.path.join(train_neg_dir, f) for f in os.listdir(train_neg_dir) if f.endswith(".txt")]
test_pos_files  = [os.path.join(test_pos_dir, f) for f in os.listdir(test_pos_dir) if f.endswith(".txt")]
test_neg_files  = [os.path.join(test_neg_dir, f) for f in os.listdir(test_neg_dir) if f.endswith(".txt")]

pos_model = UnigramModel(
    tokenize=util.nltk_tokenize,
    tokenizer_kwargs={},
    vocab=vocab,
    unk_token=unk_token,
    train_paths=train_pos_files,
    smooth="add-1",     
    lower=True
)

neg_model = UnigramModel(
    tokenize=util.nltk_tokenize,
    tokenizer_kwargs={},
    vocab=vocab,
    unk_token=unk_token,
    train_paths=train_neg_files,
    smooth="add-1",
    lower=True
)

In [20]:
models_dict = {
    "positive": pos_model,
    "negative": neg_model
}

train_data_dict = {
    "positive": train_pos_files,
    "negative": train_neg_files
}

prior_dict = get_prior(train_data_dict, lambda text: util.nltk_tokenize(text, {}))
print("Priors:", prior_dict)

Priors: {'positive': 0.5038374635403456, 'negative': 0.49616253645965436}


In [23]:
posterior_pos = get_posterior(models_dict, test_pos_files, "positive", prior_dict).reset_index(drop=True)

In [24]:
pos_pred_df = classify(posterior_pos).reset_index(drop=True)
print("Positive test accuracy:", (pos_pred_df['pred'] == pos_pred_df['class']).sum() / len(pos_pred_df))

Positive test accuracy: 0.7488


In [25]:
posterior_neg = get_posterior(models_dict, test_neg_files, "negative", prior_dict).reset_index(drop=True)

In [26]:
neg_pred_df = classify(posterior_neg).reset_index(drop=True)
print("Negative test accuracy:", (neg_pred_df['pred'] == neg_pred_df['class']).sum() / len(neg_pred_df))

Negative test accuracy: 0.87024


In [27]:
# Combine positive and negative predictions
all_pred_df = pd.concat([pos_pred_df, neg_pred_df]).reset_index(drop=True)

# Evaluation
overall_accuracy = (all_pred_df['pred'] == all_pred_df['class']).sum() / len(all_pred_df)
print("Overall test accuracy:", overall_accuracy)
precision = (all_pred_df[(all_pred_df['pred'] == 'positive') & (all_pred_df['class'] == 'positive')].shape[0]) / (all_pred_df[all_pred_df['pred'] == 'positive'].shape[0])
recall = (all_pred_df[(all_pred_df['pred'] == 'positive') & (all_pred_df['class'] == 'positive')].shape[0]) / (all_pred_df[all_pred_df['class'] == 'positive'].shape[0])
f1 = 2 * (precision * recall) / (precision + recall)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)

Overall test accuracy: 0.80952
Precision: 0.8523037698051357
Recall: 0.7488
F1 Score: 0.797206370837237


These results demonstrate that the model performs well as a simple baseline: when it predicts a review as positive, it is usually correct, but it tends to miss a number of actual positive reviews. This imbalance suggests the model is conservative in assigning positive sentiment.

## Part 4 (optional): Build Bigram Bayesian Sentiment Classifier
Add as many code and markdown chunks as is helpful

In [29]:
from BigramModel import BigramModel

pos_bigram_model = BigramModel(
    tokenize=util.nltk_tokenize,
    tokenizer_kwargs={},
    vocab=vocab,
    unk_token=unk_token,
    train_paths=train_pos_files,
    smooth="add-1",     
    lower=True
)

neg_bigram_model = BigramModel(
    tokenize=util.nltk_tokenize,
    tokenizer_kwargs={},
    vocab=vocab,
    unk_token=unk_token,
    train_paths=train_neg_files,
    smooth="add-1",
    lower=True
)

bigram_models_dict = {
    "positive": pos_bigram_model,
    "negative": neg_bigram_model
}

In [35]:
def get_bigram_posterior(models_dict, eval_fpath, class_label, prior_dict):
    
    results = []
    for model_name, model in models_dict.items():
        if isinstance(eval_fpath, list):
            test_data = model.preprocess(eval_fpath)
        else:
            test_data = model.preprocess([eval_fpath])

        for sentid, sent in enumerate(test_data):
            if not sent:
                continue

            log_likelihood = 0.0
            for i in range(1, len(sent)):
                prev_word, word = sent[i-1], sent[i]
                prob = model.get_prob(prev_word, word)
                if prob > 0:
                    log_likelihood += math.log2(prob)
                else:
                    log_likelihood = float('-inf')
                    break

            prior_prob = prior_dict.get(model_name, 1e-10)
            log_prior = math.log2(prior_prob)

            log_posterior = log_likelihood + log_prior

            results.append({
                'sentid': sentid,
                'model': model_name,
                'likelihood': log_likelihood,
                'class': class_label,
                'prior': log_prior,
                'posterior': log_posterior
            })

    return pd.DataFrame(results)


In [36]:
bigram_posterior_pos = get_bigram_posterior(bigram_models_dict, test_pos_files, "positive", prior_dict).reset_index(drop=True)

In [37]:
bigram_pos_pred_df = classify(bigram_posterior_pos).reset_index(drop=True)
print("Bigram Positive test accuracy:", (bigram_pos_pred_df['pred'] == bigram_pos_pred_df['class']).sum() / len(bigram_pos_pred_df))

Bigram Positive test accuracy: 0.28248


In [38]:
bigram_posterior_neg = get_bigram_posterior(bigram_models_dict, test_neg_files, "negative", prior_dict).reset_index(drop=True)
bigram_neg_pred_df = classify(bigram_posterior_neg).reset_index(drop=True)
print("Bigram Negative test accuracy:", (bigram_neg_pred_df['pred'] == bigram_neg_pred_df['class']).sum() / len(bigram_neg_pred_df))

Bigram Negative test accuracy: 0.91712


In [41]:
# Combine positive and negative predictions
bigram_all_pred_df = pd.concat([bigram_pos_pred_df, bigram_neg_pred_df]).reset_index(drop=True)

# Evaluation
overall_accuracy = (bigram_all_pred_df['pred'] == bigram_all_pred_df['class']).sum() / len(bigram_all_pred_df)
print("Overall test accuracy:", overall_accuracy)
precision = (bigram_all_pred_df[(bigram_all_pred_df['pred'] == 'positive') & (bigram_all_pred_df['class'] == 'positive')].shape[0]) / (bigram_all_pred_df[bigram_all_pred_df['pred'] == 'positive'].shape[0])
recall = (bigram_all_pred_df[(bigram_all_pred_df['pred'] == 'positive') & (bigram_all_pred_df['class'] == 'positive')].shape[0]) / (bigram_all_pred_df[bigram_all_pred_df['class'] == 'positive'].shape[0])
f1 = 2 * (precision * recall) / (precision + recall)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)

Overall test accuracy: 0.5998
Precision: 0.7731552441427633
Recall: 0.28248
F1 Score: 0.41378098084021797


These results indicate that while the model is fairly reliable when it predicts a review as positive, it misses the majority of actual positive reviews, leading to many false negatives. Compared to the unigram model, the bigram version performs significantly worse.